# Bluestock MF Capstone — Exploratory Data Analysis

**Deliverable:** D3 — EDA Notebook  
**Project:** Mutual Fund Analytics  
**Objective:** Explore the cleaned mutual-fund, NAV, transaction, industry, benchmark and portfolio datasets; identify trends, distributions, relationships and business insights that support the performance analytics and dashboard deliverables.

### Analysis areas
1. Data inventory and quality checks
2. Industry AUM and scheme trends
3. SIP and folio trends
4. Fund-house and category analysis
5. Risk–return relationships
6. NAV and benchmark behaviour
7. Investor transaction and demographic analysis
8. Portfolio sector exposure
9. Key findings and dashboard-ready insights

> **Important:** This notebook is descriptive EDA. Investment-performance metrics such as Sharpe, Beta and VaR are handled formally in **04_performance_analytics.ipynb**.

In [1]:
from pathlib import Path
import sqlite3
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Portable project paths: works from VS Code/Jupyter without hard-coded absolute paths.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if not (PROJECT_ROOT / "bluestock_mf.db").exists():
    PROJECT_ROOT = Path("..")
DB_PATH = PROJECT_ROOT / "bluestock_mf.db"

print("Project root:", PROJECT_ROOT.resolve())
print("Database:", DB_PATH.resolve())
print("Database exists:", DB_PATH.exists())

Project root: /mnt/data/bluestock_mf_capstone
Database: /mnt/data/bluestock_mf_capstone/bluestock_mf.db
Database exists: True


## 1. Load the SQLite data

In [2]:
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
    )["name"].tolist()

print("Tables found:")
print(tables)

Tables found:
['dim_date', 'dim_fund', 'fact_aum', 'fact_benchmark', 'fact_category_inflows', 'fact_industry_folio', 'fact_nav', 'fact_performance', 'fact_portfolio', 'fact_sip_industry', 'fact_transactions']


In [3]:
def read_table(name, parse_dates=None):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(
            f'SELECT * FROM "{name}"', conn,
            parse_dates=parse_dates or []
        )

fund = read_table("dim_fund", ["launch_date"])
date_dim = read_table("dim_date", ["date"])
nav = read_table("fact_nav", ["date"])
tx = read_table("fact_transactions", ["date"])
perf = read_table("fact_performance", ["as_of_date"])
portfolio = read_table("fact_portfolio", ["as_of_date"])
aum = read_table("fact_aum", ["quarter_end_date"])
sip = read_table("fact_sip_industry", ["month"])
cat_inflows = read_table("fact_category_inflows", ["month"])
folios = read_table("fact_industry_folio", ["as_of_date"])
benchmark = read_table("fact_benchmark", ["date"])

frames = {
    "dim_fund": fund,
    "dim_date": date_dim,
    "fact_nav": nav,
    "fact_transactions": tx,
    "fact_performance": perf,
    "fact_portfolio": portfolio,
    "fact_aum": aum,
    "fact_sip_industry": sip,
    "fact_category_inflows": cat_inflows,
    "fact_industry_folio": folios,
    "fact_benchmark": benchmark,
}

inventory = pd.DataFrame([
    {"table": k, "rows": len(v), "columns": len(v.columns), "missing_cells": int(v.isna().sum().sum())}
    for k, v in frames.items()
]).sort_values("table")

display(inventory)

,table,rows,columns,missing_cells
1,dim_date,1608,7,0
0,dim_fund,40,8,0
6,fact_aum,160,4,0
10,fact_benchmark,6900,3,0
8,fact_category_inflows,96,4,0
9,fact_industry_folio,21,5,0
2,fact_nav,45960,4,0
4,fact_performance,40,12,40
5,fact_portfolio,387,5,0
7,fact_sip_industry,48,5,0


## 2. Data-quality checks

In [4]:
quality = []
for name, df in frames.items():
    quality.append({
        "table": name,
        "rows": len(df),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_pct": round(df.isna().mean().mean() * 100, 2)
    })
quality = pd.DataFrame(quality).sort_values("missing_pct", ascending=False)
display(quality)

,table,rows,duplicate_rows,missing_cells,missing_pct
4,fact_performance,40,0,40,8.330
1,dim_date,1608,0,0,0.000
0,dim_fund,40,0,0,0.000
2,fact_nav,45960,0,0,0.000
3,fact_transactions,35039,0,0,0.000
5,fact_portfolio,387,0,0,0.000
6,fact_aum,160,0,0,0.000
7,fact_sip_industry,48,0,0,0.000
8,fact_category_inflows,96,0,0,0.000
9,fact_industry_folio,21,0,0,0.000


In [5]:
# Key integrity checks for the analytical model
checks = {
    "Unique AMFI codes in dim_fund": fund["amfi_code"].nunique(),
    "Performance rows": len(perf),
    "Performance AMFI codes": perf["amfi_code"].nunique(),
    "NAV rows": len(nav),
    "NAV funds": nav["amfi_code"].nunique(),
    "Transaction rows": len(tx),
    "Transaction investors": tx["investor_id"].nunique(),
    "Benchmark rows": len(benchmark),
}
for k, v in checks.items():
    print(f"{k}: {v:,}")

# Verify the performance table maps one row per fund.
print("\nPerformance rows equal unique funds:", len(perf) == perf["amfi_code"].nunique() == len(fund))

Unique AMFI codes in dim_fund: 40
Performance rows: 40
Performance AMFI codes: 40
NAV rows: 45,960
NAV funds: 40
Transaction rows: 35,039
Transaction investors: 5,000
Benchmark rows: 6,900

Performance rows equal unique funds: True


## 3. Fund universe overview

In [6]:
fund_summary = pd.DataFrame({
    "Metric": [
        "Funds / schemes", "Fund houses", "Categories", "Fund managers",
        "Earliest launch date", "Latest launch date"
    ],
    "Value": [
        fund["amfi_code"].nunique(),
        fund["fund_house"].nunique(),
        fund["category"].nunique(),
        fund["fund_manager"].nunique(),
        fund["launch_date"].min().date() if fund["launch_date"].notna().any() else None,
        fund["launch_date"].max().date() if fund["launch_date"].notna().any() else None,
    ]
})
display(fund_summary)

,Metric,Value
0,Funds / schemes,40
1,Fund houses,10
2,Categories,5
3,Fund managers,29
4,Earliest launch date,2015-01-22
5,Latest launch date,2019-09-21


In [7]:
category_counts = fund["category"].value_counts().reset_index()
category_counts.columns = ["category", "fund_count"]
fig = px.bar(category_counts, x="fund_count", y="category", orientation="h",
             title="Number of Funds by Category", text="fund_count")
fig.update_layout(height=550)
fig.show()

## 4. Industry AUM and scheme trends

In [8]:
aum_trend = (
    aum.groupby("quarter_end_date", as_index=False)
       .agg(aum_crore=("aum_crore", "sum"), schemes=("num_schemes", "sum"))
       .sort_values("quarter_end_date")
)

aum_trend["aum_lakh_crore"] = aum_trend["aum_crore"] / 100_000

display(aum_trend.tail(10))

,quarter_end_date,aum_crore,schemes,aum_lakh_crore
6,2023-09-30,"46,082.420",523,0.461
7,2023-12-31,"47,322.250",485,0.473
8,2024-03-31,"48,042.000",620,0.480
9,2024-06-30,"49,039.710",529,0.490
10,2024-09-30,"50,903.360",543,0.509
11,2024-12-31,"51,843.260",508,0.518
12,2025-03-31,"53,058.080",547,0.531
13,2025-06-30,"54,097.150",508,0.541
14,2025-09-30,"55,383.220",492,0.554
15,2025-12-31,"56,024.150",522,0.560


In [9]:
fig = px.line(aum_trend, x="quarter_end_date", y="aum_lakh_crore", markers=True,
              title="Industry AUM Trend (₹ lakh crore)",
              labels={"quarter_end_date":"Quarter end", "aum_lakh_crore":"AUM (₹ lakh crore)"})
fig.show()

In [10]:
latest_aum_date = aum["quarter_end_date"].max()
latest_aum = aum[aum["quarter_end_date"] == latest_aum_date].copy()
latest_aum = latest_aum.sort_values("aum_crore", ascending=False)

fig = px.bar(latest_aum.head(15), x="aum_crore", y="fund_house", orientation="h",
             title=f"Top Fund Houses by AUM — {latest_aum_date.date()}",
             labels={"aum_crore":"AUM (₹ crore)", "fund_house":"Fund house"})
fig.update_layout(yaxis={"categoryorder":"total ascending"})
fig.show()

## 5. SIP and folio trends

In [11]:
sip_monthly = sip.sort_values("month").copy()
sip_monthly["sip_inflow_lakh_crore"] = sip_monthly["sip_inflow_crore"] / 100_000

fig = go.Figure()
fig.add_trace(go.Bar(
    x=sip_monthly["month"], y=sip_monthly["sip_inflow_crore"],
    name="SIP inflow (₹ crore)"
))
fig.add_trace(go.Scatter(
    x=sip_monthly["month"], y=sip_monthly["active_sip_accounts_lakh"],
    name="Active SIP accounts (lakh)", yaxis="y2", mode="lines+markers"
))
fig.update_layout(
    title="Monthly SIP Inflows and Active SIP Accounts",
    xaxis_title="Month", yaxis_title="SIP inflow (₹ crore)",
    yaxis2=dict(title="Active SIP accounts (lakh)", overlaying="y", side="right")
)
fig.show()

In [12]:
folio_cols = [c for c in ["equity_folios_crore", "debt_folios_crore", "hybrid_folios_crore", "total_folios_crore"] if c in folios.columns]
folio_long = folios.melt(id_vars=["as_of_date"], value_vars=folio_cols,
                         var_name="folio_type", value_name="folios_crore")
fig = px.line(folio_long, x="as_of_date", y="folios_crore", color="folio_type",
              markers=True, title="Industry Folio Trend")
fig.show()

## 6. Category inflows

In [13]:
category_monthly = (
    cat_inflows.groupby("category", as_index=False)
    .agg(net_inflow_crore=("net_inflow_crore", "sum"),
         folios_lakh=("number_of_folios_lakh", "sum"))
    .sort_values("net_inflow_crore", ascending=False)
)
display(category_monthly)

fig = px.bar(category_monthly.head(15), x="net_inflow_crore", y="category", orientation="h",
             title="Top Categories by Cumulative Net Inflow",
             labels={"net_inflow_crore":"Net inflow (₹ crore)"})
fig.show()

,category,net_inflow_crore,folios_lakh
7,Small Cap,"28,732.300",364.420
4,Large Cap,"28,658.650",329.330
6,Mid Cap,"21,688.470",251.760
0,Debt,"20,133.000",244.620
1,ELSS,"15,685.540",261.190
2,Hybrid,"10,463.140",317.070
5,Liquid,"9,493.970",253.620
3,Index,"7,074.520",273.920


In [14]:
heat = cat_inflows.pivot_table(index="category", columns=cat_inflows["month"].dt.to_period("M").astype(str),
                               values="net_inflow_crore", aggfunc="sum")
fig = px.imshow(heat, aspect="auto", color_continuous_scale="RdBu_r",
                title="Category Net Inflow Heatmap")
fig.update_layout(xaxis_title="Month", yaxis_title="Category")
fig.show()

## 7. Fund performance distribution and risk–return structure

In [15]:
performance_cols = ["return_1yr_pct", "return_3yr_pct", "return_5yr_pct", "cagr_pct",
                    "sharpe_ratio", "sortino_ratio", "alpha_pct", "beta",
                    "max_drawdown_pct", "std_dev_pct"]
summary = perf[performance_cols].describe().T
summary["missing"] = perf[performance_cols].isna().sum()
display(summary)

,count,mean,std,min,25%,50%,75%,max,missing
return_1yr_pct,40.000,10.313,21.179,-37.540,-0.665,9.700,23.002,61.120,0
return_3yr_pct,40.000,5.555,14.021,-23.340,-3.250,7.500,15.198,39.850,0
cagr_pct,40.000,6.809,12.376,-22.870,-3.765,7.795,16.665,24.910,0
sharpe_ratio,40.000,-0.138,0.912,-2.980,-0.545,0.095,0.532,0.990,0
sortino_ratio,40.000,-0.235,1.554,-5.270,-0.880,0.160,0.902,1.640,0
alpha_pct,40.000,1.932,11.567,-27.080,-7.855,3.110,12.127,19.340,0
beta,40.000,-0.003,0.030,-0.060,-0.030,-0.005,0.013,0.060,0
max_drawdown_pct,40.000,-33.975,16.837,-73.120,-43.942,-30.535,-23.852,-0.390,0
std_dev_pct,40.000,20.024,6.861,0.790,18.843,19.185,21.855,29.260,0


In [16]:
perf_plot = perf.merge(
    fund[["amfi_code", "scheme_name", "fund_house", "category"]],
    on="amfi_code", how="left"
)

fig = px.scatter(
    perf_plot,
    x="std_dev_pct",
    y="return_1yr_pct",
    color="category",
    hover_name="scheme_name",
    hover_data=["fund_house", "sharpe_ratio", "beta", "max_drawdown_pct"],
    title="Risk vs 1-Year Return"
)
fig.update_layout(xaxis_title="Standard deviation (%)", yaxis_title="1-year return (%)")
fig.show()

In [17]:
fig = px.histogram(perf, x="return_1yr_pct", nbins=15, marginal="box",
                   title="Distribution of 1-Year Fund Returns",
                   labels={"return_1yr_pct":"1-year return (%)"})
fig.show()

In [18]:
fig = px.histogram(perf, x="sharpe_ratio", nbins=15, marginal="box",
                   title="Distribution of Sharpe Ratios",
                   labels={"sharpe_ratio":"Sharpe ratio"})
fig.show()

## 8. NAV behaviour and benchmark comparison

In [19]:
nav_fund = nav.merge(
    fund[["amfi_code", "scheme_name", "fund_house", "category"]],
    on="amfi_code", how="left"
)

latest_nav = (nav_fund.sort_values("date").groupby("amfi_code").tail(1)
              [["scheme_name", "fund_house", "category", "date", "nav"]]
              .sort_values("nav", ascending=False))
display(latest_nav.head(10))

,scheme_name,fund_house,category,date,nav
19532,SBI Tax Advantage Fund - Direct Growth,SBI Mutual Fund,ELSS,2026-05-29,"1,192.426"
25277,ICICI Prudential Liquid Fund - Direct Growth,ICICI Prudential Mutual Fund,Liquid,2026-05-29,"1,181.133"
41363,Kotak Liquid Fund - Direct Growth,Kotak Mahindra Mutual Fund,Liquid,2026-05-29,"1,175.652"
8042,Axis Long Term Equity Fund - Direct Growth,Axis Mutual Fund,ELSS,2026-05-29,"1,079.538"
22979,ICICI Prudential Bluechip Fund - Direct Growth,ICICI Prudential Mutual Fund,Large Cap,2026-05-29,"1,061.984"
40214,Kotak Tax Saver Fund - Direct Growth,Kotak Mahindra Mutual Fund,ELSS,2026-05-29,"1,027.855"
32171,UTI Mastershare Unit Scheme - Direct Growth,UTI Mutual Fund,Large Cap,2026-05-29,997.310
18383,SBI Liquid Fund - Direct Growth,SBI Mutual Fund,Liquid,2026-05-29,981.768
14936,SBI Bluechip Fund - Direct Plan Growth,SBI Mutual Fund,Large Cap,2026-05-29,975.182
11489,Aditya Birla Sun Life Tax Relief 96 - Direct G...,Aditya Birla Sun Life Mutual Fund,ELSS,2026-05-29,927.172


In [20]:
# Select up to five representative funds by available data (highest latest NAV here is simply a display choice).
selected_funds = latest_nav.head(5)["scheme_name"].tolist()
nav_display = nav_fund[nav_fund["scheme_name"].isin(selected_funds)].copy()
nav_display = nav_display.sort_values(["scheme_name", "date"])
nav_display["normalized_nav"] = nav_display.groupby("scheme_name")["nav"].transform(lambda s: s / s.iloc[0] * 100)

fig = px.line(nav_display, x="date", y="normalized_nav", color="scheme_name",
              title="Normalized NAV Growth — Selected Funds (Start = 100)",
              labels={"normalized_nav":"Normalized NAV", "date":"Date"})
fig.show()

In [21]:
benchmark_summary = benchmark.groupby("index_name").agg(
    start_date=("date", "min"), end_date=("date", "max"), observations=("date", "count"),
    min_close=("close_value", "min"), max_close=("close_value", "max")
).reset_index()
display(benchmark_summary)

,index_name,start_date,end_date,observations,min_close,max_close
0,BSE SmallCap,2022-01-03,2026-05-29,1150,"15,632.624","57,019.817"
1,CRISIL Gilt,2022-01-03,2026-05-29,1150,200.000,274.666
2,CRISIL Liquid,2022-01-03,2026-05-29,1150,100.000,139.153
3,Nifty 100,2022-01-03,2026-05-29,1150,"10,598.941","18,000.000"
4,Nifty 50,2022-01-03,2026-05-29,1150,"13,422.920","20,613.789"
5,Nifty Midcap 150,2022-01-03,2026-05-29,1150,"7,374.107","13,766.794"


## 9. Investor transaction EDA

In [22]:
tx_summary = pd.DataFrame({
    "Metric": [
        "Transactions", "Unique investors", "Unique funds", "States", "Cities",
        "Total transaction amount", "Average transaction amount"
    ],
    "Value": [
        len(tx), tx["investor_id"].nunique(), tx["amfi_code"].nunique(),
        tx["state"].nunique(), tx["city"].nunique(), tx["amount"].sum(), tx["amount"].mean()
    ]
})
display(tx_summary)

,Metric,Value
0,Transactions,"35,039.000"
1,Unique investors,"5,000.000"
2,Unique funds,40.000
3,States,15.000
4,Cities,16.000
5,Total transaction amount,"441,318,000.000"
6,Average transaction amount,"12,595.051"


In [23]:
state_tx = (tx.groupby("state", as_index=False)["amount"].sum()
            .sort_values("amount", ascending=False).head(15))
fig = px.bar(state_tx, x="amount", y="state", orientation="h",
             title="Top States by Transaction Amount",
             labels={"amount":"Transaction amount", "state":"State"})
fig.show()

In [24]:
type_tx = tx.groupby("transaction_type", as_index=False)["amount"].sum()
fig = px.pie(type_tx, names="transaction_type", values="amount", hole=0.45,
             title="Transaction Amount Mix")
fig.show()

In [25]:
tx_demo = tx.copy()
tx_demo["age_group"] = pd.cut(
    tx_demo["age"], bins=[0, 25, 35, 45, 55, 65, 100],
    labels=["<25", "25-35", "36-45", "46-55", "56-65", "65+"]
)
age_summary = (tx_demo.groupby("age_group", observed=False, as_index=False)
               .agg(transaction_count=("tx_id", "count"),
                    avg_amount=("amount", "mean"),
                    total_amount=("amount", "sum")))
display(age_summary)

fig = px.bar(age_summary, x="age_group", y="avg_amount",
             title="Average Transaction Amount by Age Group",
             labels={"avg_amount":"Average transaction amount"})
fig.show()

,age_group,transaction_count,avg_amount,total_amount
0,<25,3116,"11,702.343",36464500
1,25-35,8042,"12,879.756",103579000
2,36-45,8032,"12,541.770",100735500
3,46-55,8053,"12,632.932",101733000
4,56-65,7796,"12,673.935",98806000
5,65+,0,NaN,0


In [26]:
monthly_tx = (tx.assign(month=tx["date"].dt.to_period("M").dt.to_timestamp())
              .groupby("month", as_index=False)
              .agg(transaction_count=("tx_id", "count"), transaction_amount=("amount", "sum")))
fig = px.line(monthly_tx, x="month", y="transaction_count", markers=True,
              title="Monthly Transaction Volume")
fig.show()

## 10. Portfolio and sector exposure

In [27]:
sector = (portfolio.groupby("sector", as_index=False)["weight_pct"].mean()
          .sort_values("weight_pct", ascending=False))

display(sector.head(15))

fig = px.bar(sector.head(15), x="weight_pct", y="sector", orientation="h",
             title="Average Portfolio Sector Weight",
             labels={"weight_pct":"Average weight (%)"})
fig.show()

,sector,weight_pct
8,Telecom,12.117
2,Energy,11.941
6,Industrials,10.810
4,Healthcare,10.761
3,Financials,9.605
1,Consumer,9.142
0,Auto,8.772
5,IT,8.485
7,Materials,7.316


In [28]:
# Top holdings by average weight across the available portfolio snapshot.
holdings = (portfolio.groupby("stock_symbol", as_index=False)["weight_pct"].mean()
            .sort_values("weight_pct", ascending=False).head(15))
fig = px.bar(holdings, x="weight_pct", y="stock_symbol", orientation="h",
             title="Top Holdings by Average Portfolio Weight",
             labels={"weight_pct":"Average weight (%)", "stock_symbol":"Stock"})
fig.show()

## 11. Relationships and correlation checks

In [29]:
# Correlation among the supplied fund-level performance measures.
correlation_cols = [c for c in [
    "return_1yr_pct", "return_3yr_pct", "return_5yr_pct", "cagr_pct",
    "sharpe_ratio", "sortino_ratio", "alpha_pct", "beta",
    "max_drawdown_pct", "std_dev_pct"
] if c in perf.columns]

corr = perf[correlation_cols].corr()
display(corr.round(2))

fig = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale="RdBu_r",
                title="Fund Performance Metric Correlation Matrix")
fig.show()

,return_1yr_pct,return_3yr_pct,return_5yr_pct,cagr_pct,sharpe_ratio,sortino_ratio,alpha_pct,beta,max_drawdown_pct,std_dev_pct
return_1yr_pct,1.000,0.640,NaN,0.560,0.400,0.400,0.560,0.190,0.340,0.030
return_3yr_pct,0.640,1.000,NaN,0.860,0.550,0.550,0.840,0.370,0.580,-0.090
return_5yr_pct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cagr_pct,0.560,0.860,NaN,1.000,0.660,0.650,1.000,0.320,0.570,0.010
sharpe_ratio,0.400,0.550,NaN,0.660,1.000,1.000,0.680,0.210,-0.070,0.590
sortino_ratio,0.400,0.550,NaN,0.650,1.000,1.000,0.670,0.210,-0.090,0.600
alpha_pct,0.560,0.840,NaN,1.000,0.680,0.670,1.000,0.310,0.520,0.080
beta,0.190,0.370,NaN,0.320,0.210,0.210,0.310,1.000,0.260,-0.120
max_drawdown_pct,0.340,0.580,NaN,0.570,-0.070,-0.090,0.520,0.260,1.000,-0.720
std_dev_pct,0.030,-0.090,NaN,0.010,0.590,0.600,0.080,-0.120,-0.720,1.000


## 12. Automated EDA findings

In [30]:
# Generate concise, data-derived findings for the final report.
latest = aum_trend.iloc[-1]
first = aum_trend.iloc[0]
aum_growth_pct = ((latest["aum_crore"] / first["aum_crore"]) - 1) * 100 if first["aum_crore"] else np.nan

best_return = perf_plot.loc[perf_plot["return_1yr_pct"].idxmax()]
best_sharpe = perf_plot.loc[perf_plot["sharpe_ratio"].idxmax()]
lowest_risk = perf_plot.loc[perf_plot["std_dev_pct"].idxmin()]
top_category = category_monthly.iloc[0]
top_state = state_tx.iloc[0]

findings = [
    f"Industry AUM changed from ₹{first['aum_crore']:,.0f} crore at the start of the available period to ₹{latest['aum_crore']:,.0f} crore at the latest period, a cumulative change of {aum_growth_pct:.1f}%.",
    f"The category with the highest cumulative net inflow in the available category dataset is {top_category['category']} with ₹{top_category['net_inflow_crore']:,.0f} crore.",
    f"The highest 1-year return in the supplied performance snapshot is {best_return['scheme_name']} at {best_return['return_1yr_pct']:.2f}%.",
    f"The highest Sharpe ratio in the supplied performance snapshot is {best_sharpe['scheme_name']} at {best_sharpe['sharpe_ratio']:.2f}.",
    f"The lowest reported standard deviation is for {lowest_risk['scheme_name']} at {lowest_risk['std_dev_pct']:.2f}%.",
    f"{top_state['state']} is the leading state by transaction amount in the available transaction dataset, with ₹{top_state['amount']:,.0f}.",
]

for i, finding in enumerate(findings, 1):
    print(f"{i}. {finding}")

1. Industry AUM changed from ₹39,116 crore at the start of the available period to ₹56,024 crore at the latest period, a cumulative change of 43.2%.
2. The category with the highest cumulative net inflow in the available category dataset is Small Cap with ₹28,732 crore.
3. The highest 1-year return in the supplied performance snapshot is SBI Tax Advantage Fund - Direct Growth at 61.12%.
4. The highest Sharpe ratio in the supplied performance snapshot is Axis Midcap Fund - Direct Plan Growth at 0.99.
5. The lowest reported standard deviation is for ICICI Prudential Liquid Fund - Direct Growth at 0.79%.
6. Tamil Nadu is the leading state by transaction amount in the available transaction dataset, with ₹32,973,500.


## 13. EDA conclusions

### Key business themes
- **Industry scale:** AUM and folio trends provide the high-level growth story for the mutual-fund industry.
- **SIP adoption:** Monthly SIP inflows and active SIP accounts help explain recurring-investment behaviour.
- **Fund differentiation:** Return, volatility, Sharpe ratio and drawdown show that high return should not be viewed in isolation.
- **Investor behaviour:** State, age group and transaction-type distributions identify concentration and transaction-pattern differences.
- **Portfolio structure:** Sector and holding weights provide a view of concentration and diversification characteristics.
- **Dashboard linkage:** The findings above directly support the four-page Power BI/Streamlit dashboard: Industry Overview, Fund Performance, Investor Analytics, and SIP & Market Trends.

### Limitations
- This notebook describes the supplied dataset; it does not claim that historical performance predicts future returns.
- The database contains a synthetic/curated capstone dataset rather than a live production investment feed.
- Formal risk calculations and portfolio optimisation should be performed in the dedicated advanced analytics deliverables.

### Next deliverable
Proceed to **D4 — `04_performance_analytics.ipynb`** for mathematically validated CAGR, Sharpe, Beta and VaR calculations.